# 1D CNN-A — Cluster Quality: How Many Clusters?

Evaluates K-Means cluster quality across K = 2 … 16 using three complementary
metrics: the elbow method (inertia), silhouette score, and Davies-Bouldin index.
Use this to validate — or update — the `N_CLUSTERS` value in `config.py`.

> **Prerequisite:** run `1dcnn_train.ipynb` first — it saves `model.pt` to
> `DATA_DIR / SYMBOL /`.

## 1. Imports

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # macOS: prevents libiomp/libomp conflict

from datetime import date

import httpx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

print(f"PyTorch {torch.__version__} | device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Config

In [ ]:
from config import Config

cfg = Config()

# ── Override defaults here before running the rest of the notebook ────────────
# cfg.MAX_BARS       = None   # load all bars (~552k)
# cfg.EPOCHS         = 30     # full training run
# cfg.N_CLUSTERS     = 12     # try more/fewer clusters
# cfg.LATENT_DIM     = 64     # larger latent space
# cfg.N_SAMPLE       = 5_000  # render more windows in Section 9

# Expose all config fields as module-level names so every downstream cell
# can use SYMBOL, WINDOW_SIZE, LR, feature_cols, DEVICE, etc. unchanged.
globals().update(vars(cfg))

print(f"Symbol={SYMBOL}  Timeframe={TIMEFRAME}  {START_DATE} → {END_DATE}")
print("Using device:", DEVICE)

## 3. Fetch Data from Alpaca API
Calls the local `alpaca_api` FastAPI service (must be running: `uv run main.py`).
Fetches TSLA 1-minute bars and saves to both DB and CSV.

In [ ]:
if FETCH_DATA:
    params = {
        "symbols": SYMBOL,
        "timeframe": TIMEFRAME,
        "start": START_DATE,
        "end": END_DATE,
        "save_to": "db,csv",
    }
    with httpx.Client(timeout=None) as client:
        r = client.get(f"{API_BASE}/bars", params=params)
        r.raise_for_status()
        result = r.json()
    bars = result.get("data", {}).get("bars", {}).get(SYMBOL, [])
    print(f"Fetched {len(bars)} bars for {SYMBOL}")
    print("Saved:", result.get("saved"))
else:
    print("FETCH_DATA=False — skipping. Set True in Config to re-pull.")

## 4. Load Data

In [ ]:
csv_path = os.path.join(DATA_DIR, SYMBOL, f"{TIMEFRAME}.csv")
df = pd.read_csv(csv_path, parse_dates=["timestamp"], nrows=MAX_BARS)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(df):,} bars from {df['timestamp'].min()} to {df['timestamp'].max()}")
max_bars_display = 'all' if MAX_BARS is None else f'{MAX_BARS:,}'
print(f"(MAX_BARS={max_bars_display})")
df[["timestamp", "open", "high", "low", "close", "volume"]].tail()

Check for:

Duplicate timestamps
Missing timestamps (gaps)
NaNs
Infinite values
Bad OHLC relationships (high < low, etc.)

Typical checks:

In [ ]:
df = df.drop_duplicates(subset=["timestamp"])
df = df.dropna()

df.isnull().sum()
df.duplicated(subset=["timestamp"]).sum()
df[["timestamp", "open", "high", "low", "close", "volume"]].tail()

## 5. Verify Time Continuity
A CNN assumes a consistent sequence.

Look for:

missing bars
duplicate bars
irregular spacing

If you're using 1-minute candles, every row should be exactly 1 minutes apart.

In [ ]:
delta = df["timestamp"].diff()
dt = pd.to_timedelta(delta).dt.total_seconds()
print("Average time delta (seconds):", dt.mean())
print("Time delta distribution (seconds):")
print(dt.describe())

## 6. Add Features

In [ ]:
# EMAs
df["ema_9"] = df["close"].ewm(span=9, adjust=False).mean()
df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()
df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()

# MACD components
df["macd_12"] = df["close"].ewm(span=12, adjust=False).mean()
df["macd_26"] = df["close"].ewm(span=26, adjust=False).mean()

# MACD line
df["macd"] = df["macd_12"] - df["macd_26"]

# MACD signal line (9 EMA of MACD)
df["macd_9"] = df["macd"].ewm(span=9, adjust=False).mean()

# MACD histogram (optional but commonly used)
df["macd_hist"] = df["macd"] - df["macd_9"]

# Candle details
df["body"] = df["close"] - df["open"]
df["upper_wick"] = df["high"] - df[["open", "close"]].max(axis=1)
df["lower_wick"] = df[["open", "close"]].min(axis=1) - df["low"]



# other
df["return"] = df["close"].pct_change()
df["vol_return"] = df["volume"].pct_change()
df["log_return"] = np.log(df["close"] / df["close"].shift(1))
df["volume_ratio"] = (
    df["volume"] /
    df["volume"].rolling(20).mean()
)

# display sample of new features
# df[["timestamp", "close", "ema_9", "ema_21", "ema_50", "macd", "macd_9", "macd_hist", "body", "upper_wick", "lower_wick", "return", "vol_return", "log_return", "volume_ratio"]].tail()

df[df["body"] != 0][
    [
        "timestamp",
        "close",
        "ema_9",
        "ema_21",
        "ema_50",
        "macd",
        "macd_9",
        "macd_hist",
        "body",
        "upper_wick",
        "lower_wick",
        "return",
        "vol_return",
        "log_return",
        "volume_ratio",
    ]
].tail()

## 6. Remove Initial NaNs

Feature engineering creates NaNs.

In [ ]:
df = df.dropna().reset_index(drop=True)

## 7. Scale Features

This is critical.

CNNs train poorly on:

close = 45000
volume = 10000000
return = 0.001

all mixed together.

StandardScaler

Most common:

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

# feature_cols is defined in config.py — edit it there to change which features are used
# scaler = StandardScaler()
# df[feature_cols] = scaler.fit_transform(df[feature_cols])

scaler = RobustScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])
# Often better for financial data due to outliers

df[["timestamp", "open", "high", "low", "close", "volume", "ema_9", "ema_21", "ema_50", "macd", "macd_9", "macd_hist", "body", "upper_wick", "lower_wick", "return", "vol_return", "log_return", "volume_ratio"]].tail()

## 8. Create Fixed-Length Windows

A CNN does not ingest an entire dataframe.

It ingests samples.

In [ ]:
n_features = len(feature_cols)
print(f"Features ({n_features}):", feature_cols)
print("Data shape:", df[feature_cols].shape)

data = df[feature_cols].to_numpy(dtype=np.float32)

X_raw = np.lib.stride_tricks.sliding_window_view(
    data,
    window_shape=WINDOW_SIZE,
    axis=0
).transpose(0, 2, 1)   # → (N, WINDOW_SIZE, n_features)

print("X_raw shape:", X_raw.shape)

## 11. Filter Gap Windows
A window that spans an overnight or weekend gap mixes pre-gap and post-gap bars — the CNN would learn noise, not patterns. Any window whose 64-bar span crosses a gap > 5 minutes is dropped.

In [ ]:
diffs_sec = df["timestamp"].diff().dt.total_seconds().fillna(0).to_numpy()
gap_positions = np.where(diffs_sec > 300)[0]   # > 5 min between consecutive bars

valid_mask = np.ones(len(X_raw), dtype=bool)
for gp in gap_positions:
    lo = max(0, gp - WINDOW_SIZE + 1)
    hi = min(len(X_raw), gp + 1)
    valid_mask[lo:hi] = False

X_clean = X_raw[valid_mask]
print(f"Gap positions: {len(gap_positions)}")
print(f"Removed {(~valid_mask).sum():,} gap-spanning windows")
print(f"Clean windows: {X_clean.shape[0]:,}  shape: {X_clean.shape}")

## 13. Autoencoder Model
Encoder compresses `(batch, 14, 64)` → latent vector `(batch, LATENT_DIM)`.
Decoder reconstructs `(batch, 14, 64)` from the latent vector.
Training loss is reconstruction MSE — no labels needed.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # → (batch, 32, 32)
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # → (batch, 64, 16)
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # → (batch, 128, 8)
        )
        self.fc = nn.Linear(128 * 8, latent_dim)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        return self.fc(h)


class Decoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128 * 8)
        self.deconv = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=4, stride=2, padding=1),   # → (batch, 64, 16)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=4, stride=2, padding=1),    # → (batch, 32, 32)
            nn.ReLU(),
            nn.ConvTranspose1d(32, n_features, kernel_size=4, stride=2, padding=1),  # → (batch, 14, 64)
        )

    def forward(self, z):
        h = self.fc(z).view(z.size(0), 128, 8)
        return self.deconv(h)


class ConvAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim):
        super().__init__()
        self.encoder = Encoder(n_features, latent_dim)
        self.decoder = Decoder(n_features, latent_dim)

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")
print(model)

In [ ]:
# WindowDataset is needed by the latent-extraction DataLoader (Section 15)
class WindowDataset(Dataset):
    def __init__(self, X):
        self.X = X
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx]   # input == reconstruction target


In [ ]:
model_path = os.path.join(DATA_DIR, SYMBOL, "model.pt")

model = ConvAutoencoder(n_features=n_features, latent_dim=LATENT_DIM).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE))
model.eval()

print(f"Loaded  : {model_path}")
print(f"  Architecture  : ConvAutoencoder(n_features={n_features}, latent_dim={LATENT_DIM})")
print(f"  Input shape   : (batch, {n_features}, {WINDOW_SIZE})  — channels-first")
print(f"  Device        : {DEVICE}")
print(f"  Parameters    : {sum(p.numel() for p in model.parameters()):,}")

## 15. Extract Latent Vectors

In [ ]:
all_loader = DataLoader(WindowDataset(
    torch.tensor(X_clean).permute(0, 2, 1)
), batch_size=BATCH_SIZE, shuffle=False)

model.eval()
Z_list = []
with torch.no_grad():
    for batch in all_loader:
        Z_list.append(model.encoder(batch.to(DEVICE)).cpu().numpy())

Z = np.concatenate(Z_list)   # (N_clean, LATENT_DIM)
print(f'Latent matrix Z: {Z.shape}')

## 18. Cluster Quality Metrics

Three metrics, each measuring a different aspect of cluster quality:

| Metric | What it measures | Best value |
|--------|-----------------|------------|
| **Inertia (elbow)** | Total within-cluster variance — lower = tighter clusters | Look for the *bend* (elbow) |
| **Silhouette score** | How similar each point is to its own cluster vs. neighbours | Higher = better (max 1.0) |
| **Davies-Bouldin index** | Average ratio of within-cluster scatter to between-cluster distance | Lower = better (min 0.0) |

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

K_RANGE = range(2, 17)
inertias, silhouettes, db_scores = [], [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(Z)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(Z, labels, sample_size=5000, random_state=42))
    db_scores.append(davies_bouldin_score(Z, labels))
    print(f'  K={k:2d}  inertia={km.inertia_:,.0f}  silhouette={silhouettes[-1]:.4f}  DB={db_scores[-1]:.4f}')

k_vals = list(K_RANGE)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Elbow (inertia)
ax = axes[0]
ax.plot(k_vals, inertias, 'o-', color='royalblue', lw=2)
ax.set_title('Elbow Method — Inertia', fontsize=12)
ax.set_xlabel('Number of clusters K')
ax.set_ylabel('Inertia (within-cluster sum of squares)')
ax.grid(alpha=0.3)

# 2. Silhouette
best_sil_k = k_vals[int(np.argmax(silhouettes))]
ax = axes[1]
ax.plot(k_vals, silhouettes, 'o-', color='seagreen', lw=2)
ax.axvline(best_sil_k, color='tomato', ls='--', lw=1.5, label=f'peak K={best_sil_k}')
ax.set_title('Silhouette Score (higher = better)', fontsize=12)
ax.set_xlabel('Number of clusters K')
ax.set_ylabel('Silhouette score')
ax.legend(); ax.grid(alpha=0.3)

# 3. Davies-Bouldin
best_db_k = k_vals[int(np.argmin(db_scores))]
ax = axes[2]
ax.plot(k_vals, db_scores, 'o-', color='darkorange', lw=2)
ax.axvline(best_db_k, color='tomato', ls='--', lw=1.5, label=f'best K={best_db_k}')
ax.set_title('Davies-Bouldin Index (lower = better)', fontsize=12)
ax.set_xlabel('Number of clusters K')
ax.set_ylabel('Davies-Bouldin index')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(f'Cluster Quality Metrics — TSLA {TIMEFRAME} Latent Space', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Silhouette peak at K={best_sil_k}')
print(f'Davies-Bouldin best at K={best_db_k}')
print(f'Current config N_CLUSTERS={N_CLUSTERS}')

## 19. Recommendation

### How to pick K
The three metrics rarely all agree. Use them together:

1. **Elbow** — find where the inertia curve bends and stops dropping steeply.
   The K at the bend is where you get most of the benefit of clustering.

2. **Silhouette** — the peak K means windows are most clearly separated from
   their neighbours. Prefer this if you want well-defined, distinct clusters.

3. **Davies-Bouldin** — the minimum K means clusters are tight relative to
   their distance from each other. Agrees with silhouette when K is a clear choice.

**If all three point to the same K:** that's your answer — update `N_CLUSTERS` in `config.py`.

**If they disagree:** the data doesn't have a single obvious cluster count. Choose
the K that makes the most interpretable `latent_cluster.ipynb` centroid plots.